## Identifying R2Tg 25S integration locations with long-read sequencing ##
This notebook analyzes ONT amplicon long-read sequencing data to identify candidate genomic 25S loci targeted by R2Tg insertions. It filters raw reads, builds a trimmed 25S reference database, aligns reads to that database, and summarizes the chromosome-level distribution of best-supported insertions.

### Required inputs
- Raw long-read FASTQ file(s) for each sample or replicate
- Reference genome FASTA used to define 25S loci and chromosome coordinates

### Required software
- Python packages used below: `pandas`, `numpy`, `matplotlib`, `seaborn`, `biopython`
- Command-line tools used by the notebook: `cutadapt`, `seqkit`, `vsearch`
- `barrnap` is documented in the README and must be run before opening this notebook to generate `reference/genome.gff`

### How to run this notebook
- Run the cells from top to bottom from the repository root
- Edit the input file-path variables in the config cell before running the workflow
- Generate `reference/genome.gff` first by following the README instructions for the Barrnap step
- If you are analyzing multiple replicates, repeat the sample-specific filtering and alignment steps for each input FASTQ, then summarize the resulting `hits_*.tsv` files together in Step 5

### Outputs produced by the workflow
- Step 2: filtered read files in FASTQ and FASTA format
- Step 3: a FASTA database of R2-compatible 25S loci
- Step 4: `vsearch` alignment tables listing read-to-locus matches
- Step 5: per-read best-hit tables and chromosome-level summary tables
- Steps 7-8: PDF figures summarizing chromosome-level enrichment and genomic site positions

### Notes before starting
- The primer, payload, and motif sequences in later steps are hard-coded to match the experimental design in "Optimized R2 Retroelement Complexes Enable Precise and Efficient DNA Insertion into Plant Genomes" and should be updated if you use a different construct, payload, or primer set.


## Step 1: Input files and imports ##
Set the input paths to match the repository layout described in the README. This notebook assumes that `reference/genome.gff` has already been generated using the Barrnap instructions in the README. If you have multiple FASTQ files, process one file at a time by updating `RAW_FASTQ_PATH` and rerunning Steps 2-4 for each sample before continuing to Step 5.

In [ ]:
# Imports
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
import re
from pathlib import Path
from typing import Optional
from collections import Counter, OrderedDict, defaultdict
import os
import numpy as np
import matplotlib as mpl

# Notebook-wide configuration
# Only edit the input paths below.
RAW_FASTQ_PATH = "data/ONT_A.fastq"      # Example ONT FASTQ path inside the repository
GENOME_FASTA_PATH = "reference/genome.fasta"
GFF_PATH = "reference/genome.gff"
N_CHR = 19

Output file paths are derived automatically from the FASTQ filename and do not need to be edited.

In [ ]:
def derive_sample_name(path_str: str) -> str:
    if not path_str:
        return "sample"
    name = Path(path_str).name
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq", ".fasta", ".fa"):
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return Path(path_str).stem

SAMPLE_NAME = derive_sample_name(RAW_FASTQ_PATH)
OUTPUT_DIR = Path(RAW_FASTQ_PATH).expanduser().resolve().parent if RAW_FASTQ_PATH else Path(".")

FILTERED_FASTQ = str(OUTPUT_DIR / f"{SAMPLE_NAME}.filtered.fastq")
FILTERED_FASTA = str(OUTPUT_DIR / f"{SAMPLE_NAME}.filtered.fasta")
ALIGNMENT_TSV = str(OUTPUT_DIR / f"hits_{SAMPLE_NAME}.tsv")

PLUS2KB_FASTA = str(OUTPUT_DIR / "25S_plus2kb.fasta")
TRIMMED_25S_FASTA = str(OUTPUT_DIR / "25S_sequences_trimmed.fa")

BEST_HITS_TSV = str(OUTPUT_DIR / "best_or_tied_best_hits.tsv")
BEST_HITS_READ_CHRSET_TSV = str(OUTPUT_DIR / "best_or_tied_best_read_chrset.tsv")
PER_FILE_LONG_TSV = str(OUTPUT_DIR / "pct_reads_by_mapping_location.best_only.per_file.long.tsv")
PER_FILE_MATRIX_TSV = str(OUTPUT_DIR / "pct_reads_by_mapping_location.best_only.per_file.matrix.tsv")
SUMMARY_TSV = str(OUTPUT_DIR / "pct_reads_by_mapping_location.best_only.mean_sd.tsv")
FOR_PLOTTING_TSV = str(OUTPUT_DIR / "for_plotting.tsv")
FIGURE_PDF = str(OUTPUT_DIR / "figure.pdf")
CHROM_MAP_PDF = str(OUTPUT_DIR / "chrom_map.pdf")
VSEARCH_TSV_GLOB = str(OUTPUT_DIR / "hits_*.tsv")

## Step 2: Filtering and file conversion ##
This step filters raw reads to retain only those containing the expected primer and payload-adjacent sequences, and keeps reads between 2500 and 3500 bp. The filtered FASTQ output is then converted to FASTA for the alignment steps below. Update the hard-coded primer and payload-end sequences if you are analyzing a different experimental design. Repeat this step for each FASTQ file. 

In [ ]:
! python3 -m cutadapt -e 0.05 --overlap 100 --revcomp \
  -b GCAGCCACAAGCCAGTTAGGTAGCGGATAGTAGGTAGGAACAGACTTTTACTATTTCATAACGCGTCAATTACCACCTGATTTGGACCAATTCACGGGATTTGTCCAAGGTGGACGGGCCACCTTTACTTAACCCGGAAAAGGAACATATATAATTTATGTGTGTTCGATAAATAGC \
  --discard-untrimmed --action=none \
  -m 2500 -M 3500 \
  -o - "{RAW_FASTQ_PATH}" \
| python3 -m cutadapt -e 0.05 --overlap 18 \
  -b TGGTGGAAAGTCGGGGTGGT \
  --discard-untrimmed --action=none \
  -o "{FILTERED_FASTQ}" \
  - > cutadapt.log
! seqkit fq2fa "{FILTERED_FASTQ}" > "{FILTERED_FASTA}"

## Step 3: Create the database of 25S loci ##
Using the precomputed Barrnap annotation at `reference/genome.gff`, this step builds a database of candidate 25S loci. The loci are then trimmed to match the experimental amplicon design by keeping the region from the R2 insertion site to the reverse-primer binding site and adding the payload 3' sequence to the start of each record.

This step starts from the files prepared in the README. Before running the next cell, confirm that the following inputs exist:

- `reference/genome.fasta`
- `reference/genome.gff`

The notebook does not run Barrnap directly; it only consumes the Barrnap output generated during the README setup.

In [ ]:
required_inputs = [RAW_FASTQ_PATH, GENOME_FASTA_PATH, GFF_PATH]
missing_inputs = [p for p in required_inputs if not Path(p).exists()]

if missing_inputs:
    raise FileNotFoundError(
        "Missing required input files: " + ", ".join(str(p) for p in missing_inputs)
    )

print("Found required inputs:")
for p in required_inputs:
    print(f"- {p}")

In [ ]:
gff_path    = GFF_PATH           # Barrnap output generated during the README setup
genome_fa   = GENOME_FASTA_PATH
out_fa      = PLUS2KB_FASTA
DOWNSTREAM  = 2000         # extend each 25S call by 2 kb in the feature's 3' direction
TARGET_NAME = "28S_rRNA"   # Barrnap label used for 25S loci

def parse_attrs(attr_str: str) -> dict:
    d = {}
    for part in attr_str.strip().split(";"):
        if not part:
            continue
        if "=" in part:
            k, v = part.split("=", 1)
            d[k] = v
    return d

# Load the genome once so each Barrnap interval can be sliced directly by contig name.
genome = SeqIO.to_dict(SeqIO.parse(genome_fa, "fasta"))
contig_len = {k: len(v.seq) for k, v in genome.items()}

hits = 0
with open(out_fa, "w") as out, open(gff_path) as f:
    for line in f:
        if not line.strip() or line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) != 9:
            continue

        seqid, source, ftype, start, end, score, strand, phase, attrs = cols
        if ftype != "rRNA":
            continue

        a = parse_attrs(attrs)
        if a.get("Name") != TARGET_NAME:
            continue

        start = int(start)
        end = int(end)

        if seqid not in genome:
            raise KeyError(f"Contig '{seqid}' from GFF not found in genome FASTA")

        # Extend in the 3' direction
        if strand == "+":
            new_start = start
            new_end   = min(contig_len[seqid], end + DOWNSTREAM)
        elif strand == "-":
            new_start = max(1, start - DOWNSTREAM)
            new_end   = end
        else:
            continue

        seq = genome[seqid].seq[new_start - 1 : new_end]  # convert 1-based GFF coordinates to Python slicing

        # Reverse-complement minus-strand loci so every output record is 5'->3'.
        if strand == "-":
            seq = seq.reverse_complement()

        feat_id = a.get("ID", f"{TARGET_NAME}_{hits+1}")
        header = f"{feat_id}|{seqid}:{new_start}-{new_end}({strand})|downstream={DOWNSTREAM}"
        out.write(f">{header}\n")

        s = str(seq)
        for i in range(0, len(s), 80):
            out.write(s[i:i+80] + "\n")

        hits += 1

print(f"Wrote {hits} sequences to: {out_fa}")


In [ ]:
IN_FASTA  = PLUS2KB_FASTA
OUT_FASTA = TRIMMED_25S_FASTA

# Motif used to locate the R2 insertion region within each candidate 25S locus.
LOCATOR_MOTIF = "CGGGTAAACGGCGGGAGTAACTATGACTCTCTTAAGGTAGCC"  # Includes the recognition site and donor homology sequence.
CUT_AT = "TAGCC"               # Keep sequence starting at this sub-motif within the locator sequence.

# Prefix added to mimic the expected 3' payload-adjacent sequence in the amplicon.
PREFIX = "GCAGCCACAAGCCAGTTAGGTAGCGGATAGTAGGTAGGAACAGACTTTTACTATTTCATAACGCGTCAATTACCACCTGATTTGGACCAATTCACGGGATTTGTCCAAGGTGGACGGGCCACCTTTACTTAACCCGGAAAAGGAACATATATAATTTATGTGTGTTCGATAAA" 

# Right boundary of the expected amplicon; sequence after this motif is discarded.
END_MOTIF = "TGGTGGAAAGTCGGGGTGGT"  # Keep the motif itself, trim everything downstream.

# Final amplicon-length filter used for both reads and reference-derived sequences.
MIN_LEN = 2500
MAX_LEN = 3500

# Control how permissive the trimming step is when expected motifs are absent.
KEEP_UNCHANGED_IF_NO_LOCATOR = False   # If False, drop loci that do not contain the insertion-region motif.
KEEP_UNCHANGED_IF_NO_ENDMOTIF = True   # If False, drop loci missing the reverse-primer-side motif.

def trim_before_tagcc_using_locator(seq: str, locator: str, cut_at: str, prefix: str):
    s = seq.upper()
    i = s.find(locator)
    if i == -1:
        return seq, False, False

    loc_sub = s[i:i + len(locator)]
    j = loc_sub.find(cut_at)
    if j == -1:
        return seq, False, False

    cut_idx = i + j  # Absolute index of the retained insertion-site boundary.
    trimmed = seq[cut_idx:]          # Keep only the sequence from the insertion boundary onward.
    new_seq = prefix + str(trimmed)  # Prepend the expected payload-adjacent sequence.
    return new_seq, True, True

def trim_after_motif(seq: str, motif: str, use_last: bool = False):
    s = seq.upper()
    k = s.rfind(motif) if use_last else s.find(motif)
    if k == -1:
        return seq, False, False
    end = k + len(motif)
    return seq[:end], True, True

n_total = n_written = 0
n_no_locator = n_no_endmotif = 0
n_locator_edited = n_endmotif_trimmed = 0
n_len_filtered = 0

with open(OUT_FASTA, "w") as out_handle:
    for rec in SeqIO.parse(IN_FASTA, "fasta"):
        n_total += 1
        seq_str = str(rec.seq)

        # Step 1: locate the insertion-region motif, cut at the insertion boundary, and prepend the payload-side prefix.
        seq_step1, did_edit, found_locator = trim_before_tagcc_using_locator(
            seq_str, LOCATOR_MOTIF, CUT_AT, PREFIX
        )

        if not found_locator:
            n_no_locator += 1
            if KEEP_UNCHANGED_IF_NO_LOCATOR:
                # Apply the same final length filter even when keeping an unedited sequence.
                if MIN_LEN <= len(rec.seq) <= MAX_LEN:
                    SeqIO.write(rec, out_handle, "fasta")
                    n_written += 1
                else:
                    n_len_filtered += 1
            continue

        if did_edit:
            n_locator_edited += 1

        # Step 2: trim to the reverse-primer-side boundary while keeping the motif itself.
        seq_step2, did_trim, found_end = trim_after_motif(seq_step1, END_MOTIF, use_last=False)

        if not found_end:
            n_no_endmotif += 1
            if not KEEP_UNCHANGED_IF_NO_ENDMOTIF:
                continue

        if did_trim:
            n_endmotif_trimmed += 1

        # Step 3: keep only reference-derived amplicons in the expected size range.
        L = len(seq_step2)
        if not (MIN_LEN <= L <= MAX_LEN):
            n_len_filtered += 1
            continue

        rec.seq = Seq(seq_step2)
        SeqIO.write(rec, out_handle, "fasta")
        n_written += 1

print(f"Total records:                 {n_total}")
print(f"Written:                       {n_written}")
print(f"Missing locator (CTTA...):     {n_no_locator}")
print(f"Locator-cut + prefix applied:  {n_locator_edited}")
print(f"Missing end motif:             {n_no_endmotif}")
print(f"Trimmed after end motif:       {n_endmotif_trimmed}")
print(f"Filtered by length ({MIN_LEN}-{MAX_LEN}): {n_len_filtered}")
print(f"Output:                        {OUT_FASTA}")



## Step 4: Align reads with the 25S database ##
This step uses `vsearch` to align each filtered read against the trimmed 25S database and writes a TSV file of matches that pass the sequence identity and coverage thresholds. The proportion of reads with at least one retained hit will depend on sequencing quality and on the quality of the reference genome. After Steps 2-4 have been run for every FASTQ file, continue to Step 5 to summarize all `hits_*.tsv` outputs together.

In [ ]:
! vsearch --usearch_global "{FILTERED_FASTA}" \
  --db "{TRIMMED_25S_FASTA}" \
  --strand both \
  --id 0.98 \
  --query_cov 0.98 --target_cov 0.98 \
  --maxaccepts 0 \
  --blast6out "{ALIGNMENT_TSV}"

## Step 5: Summarize best read alignments per chromosome ##
For each read, the code selects the best alignment(s) by first maximizing percent identity (`pident`) and then, among those, maximizing alignment length (`aln_len`). If multiple alignments remain tied after both criteria, all tied best hits are retained and written to `best_or_tied_best_hits.tsv`.

The retained best hits are then collapsed to the chromosome level for each read, producing a per-read chromosome set and a corresponding mapping category such as `Chr19`, `Chr6+Chr19`, or `N chrs`. These results are written to `best_or_tied_best_read_chrset.tsv`.

Finally, the script summarizes the fraction of unique reads in each mapping category for every TSV file and reports the mean and standard deviation of these percentages across all input files.


In [ ]:
TSV_GLOB = VSEARCH_TSV_GLOB
MAX_CHR_IN_LABEL = 3

# Optionally restrict the summary to a user-defined set of chromosome categories.
# Example:
# KEEP_CATEGORIES = ["Chr19", "Chr6", "Chr6+Chr19"]
KEEP_CATEGORIES = None

# If True, categories outside KEEP_CATEGORIES are grouped into "Other".
COLLAPSE_TO_OTHER = False

# Write the summary tables to disk.
SAVE_TABLES = True

# Extract chromosome IDs from FASTA headers carried through the VSEARCH output.
# "28S_rRNA_1306|GWHFGPU00000019.1:7528400-7534163(-)|downstream=2000"
CHR_RE = re.compile(r"\bGWHFGPU0*([0-9]+)\.1\b")

def extract_chr(s: str) -> Optional[int]:
    m = CHR_RE.search(str(s))
    return int(m.group(1)) if m else None

def chr_set_to_label(chrs, max_in_label=3) -> str:
    chrs = tuple(sorted(set(chrs)))
    if len(chrs) == 0:
        return "NoHits"
    if len(chrs) > max_in_label:
        return f"{len(chrs)} chrs"
    return "+".join(f"Chr{c}" for c in chrs)

def find_tsv_paths(tsv_glob: str):
    p = Path(tsv_glob)
    parent = p.parent if str(p.parent) != "" else Path(".")
    pattern = p.name
    return sorted(parent.glob(pattern))

# Load every VSEARCH BLAST6-style result table selected by the glob above.
paths = find_tsv_paths(TSV_GLOB)
if not paths:
    raise FileNotFoundError(f"No TSVs matched: {TSV_GLOB}")

all_rows = []
for p in paths:
    df = pd.read_csv(p, sep="\t", header=None, dtype=str)

    required_cols = [0, 1, 2, 3]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{p.name} is missing expected BLAST6 columns: {missing}")

    df = df.rename(columns={
        0: "read_id",
        1: "hit",
        2: "pident",
        3: "aln_len"
    })

    df["pident"] = pd.to_numeric(df["pident"], errors="coerce")
    df["aln_len"] = pd.to_numeric(df["aln_len"], errors="coerce")

    df["chromosome"] = df["hit"].map(extract_chr)
    df["source_file"] = p.name

    all_rows.append(df[["source_file", "read_id", "hit", "chromosome", "pident", "aln_len"]])

hits = pd.concat(all_rows, ignore_index=True)

# Keep only rows with usable chromosome assignments and numeric scoring columns.
hits = hits.dropna(subset=["read_id", "chromosome", "pident", "aln_len"]).copy()
hits["chromosome"] = hits["chromosome"].astype(int)
hits = hits[(hits["chromosome"] >= 1) & (hits["chromosome"] <= N_CHR)]

if hits.empty:
    raise ValueError("No valid hits remain after parsing/filtering.")

# For each read, rank hits by percent identity first and alignment length second; keep exact ties.
hits["best_pident_for_read"] = (
    hits.groupby(["source_file", "read_id"])["pident"].transform("max")
)

best1 = hits[hits["pident"] == hits["best_pident_for_read"]].copy()

best1["best_aln_len_for_read"] = (
    best1.groupby(["source_file", "read_id"])["aln_len"].transform("max")
)

best_hits = best1[best1["aln_len"] == best1["best_aln_len_for_read"]].copy()

# Collapse tied best hits to one chromosome set per read so each read is counted once per file.
read_chrset = (
    best_hits.groupby(["source_file", "read_id"])["chromosome"]
    .apply(lambda s: tuple(sorted(set(int(x) for x in s))))
    .rename("chr_set")
    .reset_index()
)

read_chrset["mapping_location"] = read_chrset["chr_set"].apply(
    lambda chrs: chr_set_to_label(chrs, max_in_label=MAX_CHR_IN_LABEL)
)
read_chrset["n_chromosomes_hit"] = read_chrset["chr_set"].apply(len)

if read_chrset.empty:
    raise ValueError("No reads remained after best/tied-best filtering.")

# Optionally filter or collapse chromosome categories before summarizing percentages.
out_df = read_chrset.copy()

if KEEP_CATEGORIES is not None:
    if COLLAPSE_TO_OTHER:
        out_df["mapping_location"] = out_df["mapping_location"].where(
            out_df["mapping_location"].isin(KEEP_CATEGORIES),
            "Other"
        )
    else:
        out_df = out_df[out_df["mapping_location"].isin(KEEP_CATEGORIES)].copy()

if out_df.empty:
    raise ValueError("No reads remain after applying KEEP_CATEGORIES filter.")


# Convert per-file category counts into percentages of unique reads.
# The denominator is the number of unique reads remaining in each file after any filtering/collapsing.
counts_per_file = (
    out_df.groupby(["source_file", "mapping_location"])["read_id"]
    .nunique()
    .rename("n_reads")
    .reset_index()
)

totals_per_file = (
    out_df.groupby("source_file")["read_id"]
    .nunique()
    .rename("total_unique_reads")
    .reset_index()
)

counts_per_file = counts_per_file.merge(totals_per_file, on="source_file", how="left")
counts_per_file["pct_reads"] = 100.0 * counts_per_file["n_reads"] / counts_per_file["total_unique_reads"]

# Build a complete file-by-category matrix so absent categories are represented as 0%.
all_files = sorted(out_df["source_file"].unique())

if KEEP_CATEGORIES is not None and not COLLAPSE_TO_OTHER:
    all_cats = [c for c in KEEP_CATEGORIES if c in counts_per_file["mapping_location"].unique()]
elif KEEP_CATEGORIES is not None and COLLAPSE_TO_OTHER:
    wanted = list(KEEP_CATEGORIES) + ["Other"]
    all_cats = [c for c in wanted if c in counts_per_file["mapping_location"].unique()]
else:
    all_cats = sorted(counts_per_file["mapping_location"].unique())

pct_matrix = (
    counts_per_file
    .pivot(index="source_file", columns="mapping_location", values="pct_reads")
    .reindex(index=all_files, columns=all_cats)
    .fillna(0.0)
)

# Summarize category percentages across replicate files as mean ± SD.
if pct_matrix.shape[0] > 1:
    sd_vals = pct_matrix.std(axis=0, ddof=1).values
else:
    sd_vals = [0.0] * pct_matrix.shape[1]

summary = pd.DataFrame({
    "mapping_location": pct_matrix.columns,
    "mean_pct": pct_matrix.mean(axis=0).values,
    "sd_pct": sd_vals
})

if KEEP_CATEGORIES is None:
    summary = summary.sort_values("mean_pct", ascending=False).reset_index(drop=True)
else:
    order_map = {cat: i for i, cat in enumerate(pct_matrix.columns)}
    summary["__order"] = summary["mapping_location"].map(order_map)
    summary = summary.sort_values("__order").drop(columns="__order").reset_index(drop=True)

# Save the intermediate and summary tables used by later plotting steps.
if SAVE_TABLES:
    best_hits.to_csv(BEST_HITS_TSV, sep="\t", index=False)
    read_chrset.to_csv(BEST_HITS_READ_CHRSET_TSV, sep="\t", index=False)
    counts_per_file.to_csv(PER_FILE_LONG_TSV, sep="\t", index=False)
    pct_matrix.to_csv(PER_FILE_MATRIX_TSV, sep="\t")
    summary.to_csv(SUMMARY_TSV, sep="\t", index=False)


# Print compact summaries for notebook inspection.
print("\nSummary (mean ± SD across files):")
print(summary.to_string(index=False))

print("\nPer-file totals (after best/tied-best + any category filtering):")
print(totals_per_file.to_string(index=False))

## Step 6: Count the number of R2-compatible 25S sites in each chromosome ##
This step parses the trimmed 25S FASTA created in Step 3, records the chromosome associated with each R2-compatible site, and combines that information with the Step 5 summary table to create the TSV used for the Step 7 plot.

In [ ]:
FASTA_PATH = TRIMMED_25S_FASTA
PER_FILE_TSV = PER_FILE_LONG_TSV

# Calculate the theoretical chromosome distribution from the trimmed 25S reference set.
chr_re = re.compile(r"GWHFGPU0*([0-9]+)\.1")

chrom_counts = Counter()
total_fasta = 0

for record in SeqIO.parse(FASTA_PATH, "fasta"):
    m = chr_re.search(record.description)
    if not m:
        continue
    chrom = int(m.group(1))
    if 1 <= chrom <= N_CHR:
        chrom_counts[chrom] += 1
        total_fasta += 1

fasta_df = pd.DataFrame({
    "chromosome": list(range(1, N_CHR + 1)),
})
fasta_df["count"] = fasta_df["chromosome"].map(lambda c: chrom_counts.get(c, 0))
fasta_df["pct"] = fasta_df["count"].map(lambda x: 100.0 * x / total_fasta if total_fasta > 0 else 0.0)
fasta_df["source"] = "FASTA"
fasta_df["source_file"] = Path(FASTA_PATH).name

fasta_long = fasta_df[["source", "source_file", "chromosome", "pct"]].copy()

# Restrict the experimental distribution to unambiguous single-chromosome read assignments.
reads = pd.read_csv(PER_FILE_TSV, sep="\t")

# Keep only exact single-chromosome labels such as "Chr7".
reads_single = reads[reads["mapping_location"].str.fullmatch(r"Chr\d+")].copy()
reads_single["chromosome"] = reads_single["mapping_location"].str.replace("Chr", "", regex=False).astype(int)

# Build a full chromosome grid so missing chromosomes are recorded as 0% in each file.
all_files = sorted(reads_single["source_file"].unique())
grid = pd.MultiIndex.from_product([all_files, range(1, N_CHR + 1)], names=["source_file", "chromosome"]).to_frame(index=False)

reads_single = reads_single.merge(grid, on=["source_file", "chromosome"], how="right")

# Replace missing chromosome percentages with 0 after the grid expansion.
reads_single["pct_reads"] = reads_single["pct_reads"].fillna(0.0)

reads_long = reads_single.rename(columns={"pct_reads": "pct"})[["source_file", "chromosome", "pct"]].copy()
reads_long["source"] = "READS"

# Combine theoretical and experimental chromosome distributions for plotting.
combined_long = pd.concat([fasta_long, reads_long], ignore_index=True)

combined_long.to_csv(FOR_PLOTTING_TSV, sep="\t", index=False)
print(combined_long)

## Step 7: Plot chromosome-level read mapping ##
Using the chromosome-level read alignment results from Step 5 and the percentages of R2-compatible sites from Step 6, this step generates a simple comparison plot of theoretical versus experimental percentages for R2 insertions.

In [ ]:
data = pd.read_csv(FOR_PLOTTING_TSV, sep="\t")

# Use publication-friendly legend labels.
data["source"] = data["source"].replace({
    "FASTA": "Theoretical",
    "READS": "Experimental"
})

# Drop chromosomes that are absent from both the theoretical and experimental summaries.
keep_chromosomes = (
    data.groupby("chromosome")["pct"]
    .max()
    .loc[lambda s: s > 0]
    .index
)

plot_df = data[data["chromosome"].isin(keep_chromosomes)].copy()

with plt.rc_context({
    "font.family": "Arial",
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
}):

    sorted_df = plot_df.sort_values(by="chromosome")

    ax = sns.barplot(
        data=sorted_df,
        x="chromosome",
        y="pct",
        hue="source",
        width=0.8,
        capsize=0.25,
        errorbar="sd",
        palette=["white", "gainsboro"],
        edgecolor="black",
        linewidth=0.5,
        err_kws={"linewidth": 1.5},
    )

    ax.set_xlabel("Chromosome")
    ax.set_ylabel("Mapped Reads (%)")
    ax.set_ylim(0,85)
    plt.tick_params(axis="y", direction="in")
    plt.legend(title="", loc="upper center")
    plt.savefig(FIGURE_PDF)

## Step 8: Plot compatible 25S sites and supported insertions ##
This final step plots the genomic positions of every R2-compatible 25S site from the Step 3 FASTA alongside the insertion-supported positions from Step 5 `best_or_tied_best_hits.tsv`. Using the genome FASTA, it highlights which compatible sites are supported by the sequencing data. Insertion sites with multiple best-supported loci will appear multiple times.

In [ ]:
# Plot inputs and styling for the chromosome-position map.
genome_fasta = GENOME_FASTA_PATH
fasta_25S    = TRIMMED_25S_FASTA
hits_csv     = BEST_HITS_TSV
output_plot  = CHROM_MAP_PDF

x_scale         = 1e-6   # bp -> Mb
chrom_h         = 0.5
fasta_color      = "seagreen"
csv_color       = "violet"
alpha = 0.5
blue_linewidth  = 1.2
red_linewidth   = 1.2

# Parse genomic coordinates from FASTA headers carried through earlier steps.
# 28S_rRNA_71|GWHFGPU00000006.1:99744-105507(-)|downstream=2000
rx_header = re.compile(
    r"^[^|]+\|([^:|]+):(\d+)-(\d+)\(([+-])\)\|"
)

def parse_site_header(header):
    """
    Parse a header string like:
    28S_rRNA_71|GWHFGPU00000006.1:99744-105507(-)|downstream=2000

    Returns:
        contig_id, start_bp, end_bp, strand
    or
        None if parsing fails
    """
    m = rx_header.search(header)
    if not m:
        return None
    contig_id = m.group(1)
    start_bp = int(m.group(2))
    end_bp = int(m.group(3))
    strand = m.group(4)
    return contig_id, start_bp, end_bp, strand

# Load genome contigs in FASTA order so the plot rows match the reference file.
contig_order = []
contig_len = OrderedDict()

for rec in SeqIO.parse(genome_fasta, "fasta"):
    cid = rec.id
    contig_order.append(cid)
    contig_len[cid] = len(rec.seq)

if not contig_order:
    raise ValueError(f"No sequences found in genome FASTA: {genome_fasta}")

# Collect all R2-compatible 25S site positions from the trimmed FASTA.
fasta_sites_bp = defaultdict(list)
unknown_fasta_contigs = set()

for rec in SeqIO.parse(fasta_25S, "fasta"):
    parsed = parse_site_header(rec.description)
    if parsed is None:
        continue

    cid, start_bp, end_bp, strand = parsed

    if cid in contig_len:
        fasta_sites_bp[cid].append(start_bp)
    else:
        unknown_fasta_contigs.add(cid)

if unknown_fasta_contigs:
    print(f"WARNING: {len(unknown_fasta_contigs)} FASTA contig IDs not found in genome FASTA (showing up to 10):")
    print("  " + ", ".join(sorted(list(unknown_fasta_contigs))[:10]))

# Collect insertion-supported positions from the best-hit table; use a set to remove duplicates.
hits_sites_bp = defaultdict(set)
unknown_csv_contigs = set()

df = pd.read_csv(hits_csv, sep='\t')

if "hit" not in df.columns:
    raise ValueError("Input CSV must contain a 'hit' column")

for hit in df["hit"].dropna():
    parsed = parse_site_header(str(hit))
    if parsed is None:
        continue

    cid, start_bp, end_bp, strand = parsed

    if cid in contig_len:
        hits_sites_bp[cid].add(start_bp)
    else:
        unknown_csv_contigs.add(cid)

if unknown_csv_contigs:
    print(f"WARNING: {len(unknown_csv_contigs)} CSV contig IDs not found in genome FASTA (showing up to 10):")
    print("  " + ", ".join(sorted(list(unknown_csv_contigs))[:10]))

# Configure the chromosome map aesthetics.
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

n = len(contig_order)
fig_h = max(6, 0.18 * n)
fig, ax = plt.subplots(figsize=(12, fig_h))

line_half = chrom_h * 0.47
max_bp = max(contig_len.values())

rng = np.random.default_rng(42)  # Fixed seed keeps point jitter reproducible across runs.

# Draw each chromosome/contig as one horizontal row.
for row, cid in enumerate(contig_order, start=1):
    y = row
    L = contig_len[cid]

    # Background chromosome bar.
    ax.add_patch(plt.Rectangle(
        (0, y - chrom_h / 2),
        L * x_scale,
        chrom_h,
        facecolor="snow",
        edgecolor="black",
        linewidth=0.6,
        zorder=1
    ))

    # All compatible 25S sites from the trimmed reference FASTA.
    xs_fasta_mb = [bp * x_scale for bp in fasta_sites_bp.get(cid, [])]
    if xs_fasta_mb:
        ax.vlines(
            xs_fasta_mb,
            y - line_half,
            y + line_half,
            color=fasta_color,
            linewidth=blue_linewidth,
            alpha=1,
            zorder=2
        )

    # Insertion-supported sites from the best-hit table, jittered vertically for visibility.
    xs_hits_mb = [bp * x_scale for bp in sorted(hits_sites_bp.get(cid, set()))]
    if xs_hits_mb:
        jitter = rng.uniform(-chrom_h * 0.3, chrom_h * 0.3, size=len(xs_hits_mb))
        ys_hits = y + jitter

        ax.scatter(
            xs_hits_mb,
            ys_hits,
            color=csv_color,
            s=4,
            alpha=0.75,
            zorder=3, 
            edgecolors='black', 
            linewidths=0.05
        )

# Label rows numerically to match chromosome numbering.
chr_labels = [f"{i}" for i in range(1, n + 1)]

ax.set(
    ylim=(0.5, n + 0.5),
    xlim=(0, max_bp * x_scale * 1.1),
    yticks=range(1, n + 1),
    yticklabels=chr_labels,
    xlabel="Position (Mb)",
    ylabel="Chromosome"
)

ax.invert_yaxis()
ax.grid(axis="x", linestyle="--", alpha=0.3)

os.makedirs(os.path.dirname(output_plot) or ".", exist_ok=True)
plt.tight_layout()
plt.savefig(output_plot)
plt.show()